这种方式需要按照JSON Schema规范拼接JSON字符串，比较繁琐，并且缺少校验机制。不推荐。

举例1：简单例子

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

#优先加载配置
load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
        "type": "string",
        "description": "The title of the movie"
        },
        "year": {
        "type": "integer",
        "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
        "description": "The director of the movie"
        },
        "rating": {
        "type": "number",
        "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

#设置模型结构化输出
structured_llm = model_openai.with_structured_output(json_schema, method="json_schema")

response = structured_llm.invoke("给出盗梦空间的信息")

rprint(response)


{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}

举例2：嵌套结构

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

#优先加载配置
load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)
# 1. 定义嵌套的 JSON Schema
project_schema = {
    "title": "MovieInfo",
    "description": "包含电影标题、上映年份、导演、演员和评分的电影对象",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影标题"},
        "year": {"type": "integer", "description": "上映年份"},
        "director": {"type": "string", "description": "导演"},
        "cast": { # 定义嵌套数组
            "type": "array",
            "description": "演员列表",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "演员姓名"},
                    "role": {"type": "string", "description": "演员角色"}
                },
                "required": ["name", "role"]
            }
        },
        "rating": {"type": "number", "description": "评分（10分制）"}
    },
    "required": ["title", "year", "director", "cast", "rating"]
}

#设置模型结构化输出  method="json_schema" 可以不需要
structured_llm = model_openai.with_structured_output(project_schema, method="json_schema")

response = structured_llm.invoke("生成一个关于《星际穿越》的电影信息，包含导演、演员、评分")

rprint(response)

{
    'title': '星际穿越',
    'year': 2014,
    'director': '克里斯托弗·诺兰',
    'cast': [
        {'name': '马修·麦康纳', 'role': '库珀'},
        {'name': '安妮·海瑟薇', 'role': '布兰德博士'},
        {'name': '杰西卡·查斯坦', 'role': '墨菲（成年）'},
        {'name': '迈克尔·凯恩', 'role': '詹姆斯教授'}
    ],
    'rating': 9.4
}